# etl-example: tracking-survey ETL walkthrough

This notebook runs the whole pipeline **in-process**: FastAPI's `TestClient`
calls the API directly, with no server and no Docker, against a temporary
SQLite database.

Scenario:

1. Waves 1 and 2 are already in the database.
2. Wave 3 arrives with 12 rows: 8 wave 3 interviews, **one late wave 2
   interview**, and one row for each cleaning rule.
3. The pipeline validates, cleans, transforms and loads the batch, then
   recalculates the summaries of **every wave that received new rows**:
   waves 2 and 3.

In [1]:
import contextlib
import dataclasses
import tempfile
from pathlib import Path
from typing import Any

import pandas as pd
from fastapi.testclient import TestClient

from etl_example.api import create_app
from etl_example.database import define_tables, summary_query
from etl_example.settings import DEFAULT_SETTINGS, Environment
from etl_example.survey_csv import parse_survey_csv, to_batch_json

DATA_DIRECTORY = Path.cwd().parent / "data"

resources = contextlib.ExitStack()
database_directory = Path(
    resources.enter_context(tempfile.TemporaryDirectory())
)
settings = dataclasses.replace(
    DEFAULT_SETTINGS[Environment.DEV],
    database_url=f"sqlite:///{database_directory / 'example.db'}",
    log_level="WARNING",
)
client = resources.enter_context(TestClient(create_app(settings)))
client.get("/health").json()

{'status': 'ok', 'environment': 'dev'}

In [2]:
def read_wave_rows(file_name: str) -> list[dict[str, Any]]:
    """Convert a wave CSV export to the rows of the upload body."""
    csv_path = DATA_DIRECTORY / file_name
    return parse_survey_csv(csv_path.read_text(), file_name)


def upload(rows: list[dict[str, Any]]) -> dict[str, Any]:
    """POST a batch and return the batch report."""
    response = client.post("/batches", json={"respondents": rows})
    response.raise_for_status()
    return response.json()


def summary_table(wave: int) -> pd.DataFrame:
    """GET a wave summary as one row per age category."""
    response = client.get(f"/waves/{wave}/summary")
    response.raise_for_status()
    records = [
        {
            "age_category": category["age_category"],
            "respondents": category["respondents_total"],
            **{
                f"{brand['brand']}_{measure}": brand[f"{measure}_total"]
                for brand in category["brands"]
                for measure in ("awareness", "usage")
            },
        }
        for category in response.json()["age_categories"]
    ]
    return pd.DataFrame(records).set_index("age_category")

## 1. Waves 1 and 2 are already stored

In [3]:
seed_reports = {
    file_name: upload(read_wave_rows(file_name))
    for file_name in ("wave1.csv", "wave2.csv")
}
pd.DataFrame(seed_reports).T[
    ["rows_received", "rows_loaded", "refreshed_waves"]
]

,rows_received,rows_loaded,refreshed_waves
wave1.csv,12,12,[1]
wave2.csv,11,11,[2]


Wave 2 has 11 respondents so far: one interview is missing.

In [4]:
summary_table(2)

,respondents,brand1_awareness,brand1_usage,brand2_awareness,brand2_usage,brand3_awareness,brand3_usage
age_category,,,,,,,
18-24,2,2,1,1,1,1,0
25-34,3,3,3,2,1,0,0
35-44,2,2,2,1,0,1,1
45-54,2,1,0,2,1,1,1
55+,2,1,1,2,1,0,0


## 2. Wave 3 arrives

The raw export: row 5 is the late wave 2 interview (respondent 2012).
Rows 3, 7, 9 and 10 trigger the cleaning rules.

In [5]:
pd.read_csv(DATA_DIRECTORY / "wave3.csv", dtype=str, keep_default_na=False)

,respondent_id,interview_date,wave,birth_year,brand1_awareness,brand2_awareness,brand3_awareness,brand1_usage,brand2_usage,brand3_usage
0,3001,07/09/2026,3,2005,1,1,0,1,0,0
1,3002,07/09/2026,3,1997,1,1,1,1,1,0
2,3003,08/09/2026,3,1988,1,0,0,1,0,1
3,3004,08/09/2026,3,1978,1,1,0,0,1,0
4,2012,26/06/2026,2,1983,1,1,0,1,0,0
5,3005,09/09/2026,3,1964,1,1,0,1,,0
6,3006,09/09/2026,3,1992,,,,,,
7,3007,10/09/2026,3,2002,0,1,1,0,1,0
8,3008,31/09/2026,3,1991,1,0,0,1,0,0
9,1005,10/09/2026,3,1962,1,0,0,1,0,0


The CSV is converted to JSON, here are the first two rows of the request body:

In [6]:
wave3_rows = read_wave_rows("wave3.csv")
print(to_batch_json(wave3_rows[:2]))

{
  "respondents": [
    {
      "respondent_id": 3001,
      "interview_date": "07/09/2026",
      "wave": 3,
      "birth_year": 2005,
      "brand1_awareness": 1,
      "brand2_awareness": 1,
      "brand3_awareness": 0,
      "brand1_usage": 1,
      "brand2_usage": 0,
      "brand3_usage": 0
    },
    {
      "respondent_id": 3002,
      "interview_date": "07/09/2026",
      "wave": 3,
      "birth_year": 1997,
      "brand1_awareness": 1,
      "brand2_awareness": 1,
      "brand3_awareness": 1,
      "brand1_usage": 1,
      "brand2_usage": 1,
      "brand3_usage": 0
    }
  ]
}


In [7]:
report = upload(wave3_rows)
{
    key: report[key]
    for key in ("rows_received", "rows_loaded", "refreshed_waves")
}

{'rows_received': 12, 'rows_loaded': 9, 'refreshed_waves': [2, 3]}

Dropped rows, each with its row number in the batch and the reason:

In [8]:
pd.DataFrame(report["dropped_rows"])

,row_number,respondent_id,reason,detail
0,7,3006,no_answers,all awareness and usage answers are missing
1,9,3008,invalid_value,"interview_date: Value error, day 31 must be in..."
2,10,1005,duplicate_in_database,respondent ID already stored in the database


Corrected values: awareness comes first, so usage without awareness is set to 0.

In [9]:
pd.DataFrame(report["corrected_rows"])

,row_number,respondent_id,reason,detail
0,3,3003,usage_without_awareness,brand3_usage set to 0 because brand3_awareness...


## 3. Refreshed summaries

Wave 2 now has 12 respondents: the late interview was added and the wave
summary rebuilt.

In [10]:
summary_table(2)

,respondents,brand1_awareness,brand1_usage,brand2_awareness,brand2_usage,brand3_awareness,brand3_usage
age_category,,,,,,,
18-24,2,2,1,1,1,1,0
25-34,3,3,3,2,1,0,0
35-44,3,3,3,2,0,1,1
45-54,2,1,0,2,1,1,1
55+,2,1,1,2,1,0,0


In [11]:
summary_table(3)

,respondents,brand1_awareness,brand1_usage,brand2_awareness,brand2_usage,brand3_awareness,brand3_usage
age_category,,,,,,,
18-24,2,1,1,2,1,1,0
25-34,1,1,1,1,1,1,0
35-44,1,1,1,0,0,0,0
45-54,2,2,1,2,2,0,0
55+,2,2,1,1,0,1,1


## 4. The SQL behind the refresh

For every wave with rows tagged `new`, the summary rows are deleted and
re-inserted from this query. Then the rows are tagged `current`. All of
this runs in one transaction.

In [12]:
query = summary_query(define_tables(), [2, 3])
print(query.compile(compile_kwargs={"literal_binds": True}))

SELECT respondents.wave, respondents.age_category, count(*) AS respondents_total, coalesce(sum(respondents.brand1_awareness), 0) AS brand1_awareness_total, coalesce(sum(respondents.brand2_awareness), 0) AS brand2_awareness_total, coalesce(sum(respondents.brand3_awareness), 0) AS brand3_awareness_total, coalesce(sum(respondents.brand1_usage), 0) AS brand1_usage_total, coalesce(sum(respondents.brand2_usage), 0) AS brand2_usage_total, coalesce(sum(respondents.brand3_usage), 0) AS brand3_usage_total 
FROM respondents 
WHERE respondents.wave IN (2, 3) GROUP BY respondents.wave, respondents.age_category


In [13]:
resources.close()  # stop the app and delete the temporary database